# Overwatch
**| Global Solution|**

> Crie sua API key gratuita em https://console.groq.com


In [1]:
# CELULA 1 - Instalacao
# IMPORTANTE: apos executar faca Runtime > Restart session
# Depois execute da Celula 2 em diante (nao repita esta)

# fastembed usa ONNX (sem torch) - resolve o erro numpy dtype
!pip install -q \
    llama-index-core \
    llama-index-readers-file \
    llama-index-embeddings-fastembed \
    llama-index-llms-groq \
    fastembed \
    pypdf \
    gradio

print('Instalacao concluida!')
print('AGORA: Runtime > Restart session')
print('Depois execute da Celula 2 em diante')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.2/121.2 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.9/323.9 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.

In [8]:
# CELULA 2 - Configuracoes
import os

GROQ_API_KEY  = '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'  # <- substitua pela sua chave gsk_...
os.environ['GROQ_API_KEY'] = GROQ_API_KEY

EMBED_MODEL   = 'BAAI/bge-small-en-v1.5'
LLM_MODEL     = 'llama-3.3-70b-versatile'
CHUNK_SIZE    = 512
CHUNK_OVERLAP = 64
TOP_K         = 4
DATA_DIR      = '/content/data'
STORAGE_DIR   = '/content/storage'

SYSTEM_PROMPT = (
    'Voce e o ORBITAL SENTINEL, assistente especializado em dados espaciais, '
    'previsao climatica e prevencao de desastres naturais. Suas respostas devem: '
    '1) Ser baseadas EXCLUSIVAMENTE nos documentos fornecidos. '
    '2) Citar o documento de origem quando possivel. '
    '3) Indicar quando a informacao NAO estiver nos documentos. '
    '4) Conectar tecnologias espaciais com impactos na sociedade brasileira. '
    '5) Responder sempre em Portugues do Brasil.'
)

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(STORAGE_DIR, exist_ok=True)

print('Configuracoes definidas.')
print(f'  Embedding : {EMBED_MODEL} (FastEmbed/ONNX - sem conflito numpy)')
print(f'  LLM       : {LLM_MODEL} (Groq)')
print(f'  Chunk     : {CHUNK_SIZE} tokens | overlap {CHUNK_OVERLAP}')
print(f'  Top-K     : {TOP_K} chunks por consulta')


Configuracoes definidas.
  Embedding : BAAI/bge-small-en-v1.5 (FastEmbed/ONNX - sem conflito numpy)
  LLM       : llama-3.3-70b-versatile (Groq)
  Chunk     : 512 tokens | overlap 64
  Top-K     : 4 chunks por consulta


In [9]:
# CELULA 3 - Criar base de conhecimento inicial
import os

doc1 = 'Satelites e Previsao Climatica\n\n'
doc1 += 'SENSORIAMENTO REMOTO\n'
doc1 += 'O sensoriamento remoto por satelite revolucionou o monitoramento de fenomenos climaticos. '
doc1 += 'A NASA opera o Earth Observing System com TERRA e AQUA equipados com sensor MODIS, '
doc1 += 'fornecendo imagens diarias em 36 bandas espectrais com resolucao de 250m a 1km.\n\n'
doc1 += 'PREVISAO DE ENCHENTES\n'
doc1 += 'O IMERG combina dados de multiplos satelites para estimar precipitacao global '
doc1 += 'com resolucao de 10km e atualizacao a cada 30 minutos. '
doc1 += 'O satelite SMAP mede umidade do solo a cada 2-3 dias com resolucao de 9km. '
doc1 += 'Solos saturados aumentam o risco de enchentes. Integrar SMAP a modelos hidrologicos '
doc1 += 'aumenta a precisao dos alertas em ate 40 por cento. '
doc1 += 'O SRTM fornece modelo de elevacao global a 30m, essencial para mapear planicies de inundacao. '
doc1 += 'O Copernicus EMS realizou mais de 500 mapeamentos de inundacoes entre 2012 e 2024. '
doc1 += 'No Brasil foi usado nas enchentes do RS em 2024 mapeando mais de 2 milhoes de hectares.\n\n'
doc1 += 'MONITORAMENTO DE SECAS\n'
doc1 += 'O NDVI detecta vegetacao estressada com valores abaixo de 0.2. '
doc1 += 'Os satelites GRACE-FO medem variacoes gravitacionais causadas por mudancas na '
doc1 += 'distribuicao de agua, monitorando aquiferos subterraneos. Regioes com deplecao de '
doc1 += 'aquiferos sao identificadas com 6 a 18 meses de antecedencia.\n\n'
doc1 += 'DETECCAO DE QUEIMADAS\n'
doc1 += 'O sistema INPE QUEIMADAS usa MODIS e VIIRS para detectar focos ativos com 2-4 passagens diarias. '
doc1 += 'O Sentinel-2 confirma a area queimada com 10m de resolucao. '
doc1 += 'O GOES-16 permite monitoramento a cada 10 minutos sobre a America do Sul. '
doc1 += 'Algoritmos de ML alcancam 94 por cento de acuracia com latencia inferior a 3 horas.\n\n'
doc1 += 'INTELIGENCIA ARTIFICIAL\n'
doc1 += 'Os modelos GraphCast e Pangu-Weather superam modelos tradicionais do ECMWF e NOAA em previsoes de 10 dias. '
doc1 += 'LLMs integrados via RAG permitem perguntas em linguagem natural sobre protocolos e dados historicos.'

doc2 = 'Tecnologias Espaciais e Desastres no Brasil\n\n'
doc2 += 'CONTEXTO BRASILEIRO\n'
doc2 += 'O Brasil registrou mais de 38 mil ocorrencias de desastres entre 2000 e 2023, '
doc2 += '20 mil mortes e R$ 200 bilhoes em prejuizos (CEMADEN 2023). '
doc2 += 'Sul: ciclones e enchentes. Sudeste: deslizamentos. Nordeste: secas. Amazonia: incendios.\n\n'
doc2 += 'INFRAESTRUTURA ESPACIAL BRASILEIRA\n'
doc2 += 'O INPE opera PRODES (desmatamento desde 1988), DETER (deteccao quinzenal) e QUEIMADAS (desde 1998). '
doc2 += 'O CEMADEN integra mais de 4 mil pluviometros e 133 estacoes hidrologicas cobrindo 957 municipios em risco. '
doc2 += 'O satelite Amazonia-1 lancado em 2021 e o primeiro 100 por cento brasileiro, '
doc2 += 'operando a 752km revisitando a Amazonia a cada 5 dias com resolucao de 64m.\n\n'
doc2 += 'ALERTA NACIONAL\n'
doc2 += 'O tempo entre deteccao de anomalia e alerta ao cidadao caiu de 6-8 horas em 2010 '
doc2 += 'para 45-90 minutos em 2024 gracas a automacao com IA.\n\n'
doc2 += 'CONECTIVIDADE VIA SATELITE\n'
doc2 += 'As enchentes do RS em 2024 isolaram mais de 400 municipios. '
doc2 += 'O Starlink distribuiu terminais portateis pelo governo para reconectar as regioes afetadas. '
doc2 += 'BGAN Inmarsat e Iridium apoiam equipes de resgate em campo.\n\n'
doc2 += 'RAG E IA PARA DEFESA CIVIL\n'
doc2 += 'Um sistema RAG integraria historico CEMADEN, Lei 12608 de 2012, protocolos de evacuacao '
doc2 += 'e relatorios do IPCC. Alertas compreensiveis aumentam em 60 por cento a taxa de evacuacao voluntaria.'

with open(f'{DATA_DIR}/satelites.txt', 'w', encoding='utf-8') as f:
    f.write(doc1)
with open(f'{DATA_DIR}/desastres_brasil.txt', 'w', encoding='utf-8') as f:
    f.write(doc2)

print('Base de conhecimento criada em /content/data/')
print('  satelites.txt')
print('  desastres_brasil.txt')


Base de conhecimento criada em /content/data/
  satelites.txt
  desastres_brasil.txt


In [10]:
# CELULA 4 - Configurar modelos e indexar documentos
# Pode demorar 1-2 minutos (download do modelo de embedding)
import os
from llama_index.core import Settings, VectorStoreIndex, SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.fastembed import FastEmbedEmbedding
from llama_index.llms.groq import Groq

print('Carregando embedding FastEmbed/ONNX...')
Settings.embed_model = FastEmbedEmbedding(model_name=EMBED_MODEL)
print(f'OK Embedding: {EMBED_MODEL}')

Settings.llm = Groq(model=LLM_MODEL, api_key=GROQ_API_KEY, system_prompt=SYSTEM_PROMPT)
print(f'OK LLM: {LLM_MODEL}')

Settings.node_parser = SentenceSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
print(f'OK Chunking: {CHUNK_SIZE} tokens, overlap {CHUNK_OVERLAP}')

def build_index():
    docs = SimpleDirectoryReader(
        input_dir=DATA_DIR, recursive=True,
        required_exts=['.pdf', '.txt', '.md'],
    ).load_data()
    n_files = len(set(d.metadata.get('file_name','?') for d in docs))
    print(f'  {len(docs)} trecho(s) de {n_files} arquivo(s)')
    print('  Gerando embeddings e construindo FAISS...')
    idx = VectorStoreIndex.from_documents(docs, show_progress=True)
    idx.storage_context.persist(persist_dir=STORAGE_DIR)
    return idx

print('\nIndexando documentos...')
index = build_index()
query_engine = index.as_query_engine(similarity_top_k=TOP_K, streaming=False, verbose=False)
print('\nSistema RAG pronto! Execute a Celula 5 para abrir o chat.')


Carregando embedding FastEmbed/ONNX...
OK Embedding: BAAI/bge-small-en-v1.5
OK LLM: llama-3.3-70b-versatile
OK Chunking: 512 tokens, overlap 64

Indexando documentos...
  3 trecho(s) de 3 arquivo(s)
  Gerando embeddings e construindo FAISS...


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/8 [00:00<?, ?it/s]


Sistema RAG pronto! Execute a Celula 5 para abrir o chat.


In [11]:
# CELULA 5 - Interface Gradio (chat + upload de documentos)
import gradio as gr
import shutil, os

def add_document(file_obj):
    global index, query_engine
    if file_obj is None:
        return 'Nenhum arquivo enviado.'
    dest = os.path.join(DATA_DIR, os.path.basename(file_obj.name))
    shutil.copy(file_obj.name, dest)
    index = build_index()
    query_engine = index.as_query_engine(similarity_top_k=TOP_K, streaming=False, verbose=False)
    return f"'{os.path.basename(dest)}' adicionado e indexado!"

def chat(message, history):
    if not message.strip():
        return history, ''
    response = query_engine.query(message)
    answer = str(response)
    if hasattr(response, 'source_nodes') and response.source_nodes:
        answer += '\n\n---\nFontes consultadas:'
        for i, node in enumerate(response.source_nodes, 1):
            score = getattr(node, 'score', None)
            fname = node.metadata.get('file_name', 'desconhecido')
            score_str = f' (score: {score:.3f})' if score else ''
            answer += f'\n  [{i}] {fname}{score_str}'
    history.append((message, answer))
    return history, ''

with gr.Blocks(title='OVERWATCH', theme=gr.themes.Soft(primary_hue='red')) as demo:
    gr.HTML('<div style="text-align:center;padding:16px 0">'
            '<h1 style="color:#C0392B">OVERWATCH</h1>'
            '<p>Assistente RAG para Previsao Climatica e Desastres Naturais</p>'
            '<small>|Global Solution|</small>'
            '</div>')
    with gr.Tabs():
        with gr.Tab('Chat'):
            chatbot = gr.Chatbot(label='OVERWATCH', height=420, bubble_full_width=False)
            with gr.Row():
                msg_box = gr.Textbox(placeholder='Digite sua pergunta...', label='', scale=5, container=False)
                send_btn = gr.Button('Enviar', variant='primary', scale=1)
            gr.Examples(
                examples=[
                    ['Como satelites ajudam a prever enchentes?'],
                    ['Como a IA detecta queimadas via satelite?'],
                    ['Como o Brasil monitora desastres com dados espaciais?'],
                    ['O que e o satelite GRACE-FO?'],
                    ['Como o Starlink ajudou nas enchentes do RS em 2024?'],
                ],
                inputs=msg_box,
                label='Exemplos de perguntas - clique para usar'
            )
            send_btn.click(chat, [msg_box, chatbot], [chatbot, msg_box])
            msg_box.submit(chat, [msg_box, chatbot], [chatbot, msg_box])
        with gr.Tab('Adicionar Documento'):
            gr.Markdown('### Adicione documentos (PDF, TXT, MD)\nApos o upload o sistema re-indexa automaticamente.')
            upload = gr.File(label='Selecione o arquivo', file_types=['.pdf','.txt','.md'])
            upload_btn = gr.Button('Enviar e Indexar', variant='primary')
            upload_result = gr.Textbox(label='Status', interactive=False)
            upload_btn.click(add_document, inputs=upload, outputs=upload_result)
        with gr.Tab('Sobre o Sistema'):
            gr.Markdown(
                '## Pipeline RAG\n\n'
                '| Etapa | Componente | Detalhe |\n'
                '|-------|-----------|---------|\n'
                '| Leitura | SimpleDirectoryReader | PDF, TXT, MD |\n'
                '| Chunking | SentenceSplitter | 512 tokens, overlap 64 |\n'
                '| Embedding | BAAI/bge-small-en-v1.5 | FastEmbed ONNX |\n'
                '| Vector Store | FAISS (LlamaIndex) | Cosseno |\n'
                '| Retrieval | Top-K=4 | 4 chunks |\n'
                '| LLM | llama-3.3-70b-versatile | Groq API |\n'
            )
demo.launch(share=True, debug=False, quiet=True)


/tmp/ipykernel_11299/2289288115.py:30: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title='OVERWATCH', theme=gr.themes.Soft(primary_hue='red')) as demo:
/tmp/ipykernel_11299/2289288115.py:38: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label='OVERWATCH', height=420, bubble_full_width=False)
/tmp/ipykernel_11299/2289288115.py:38: DeprecationWarning: The 'bubble_full_width' parameter will be removed in Gradio 6.0. This parameter no longer has any effect.
  chatbot = gr.Chatbot(label='OVERWATCH', height=420, bubble_full_width=False)
/tmp/ipykernel_11299/2289288115.py:38: DeprecationW

* Running on public URL: https://25ae30192f85a199df.gradio.live


In [12]:
# CELULA 6 - Avaliacao quantitativa
import time

perguntas_teste = [
    'Como satelites ajudam a prever enchentes?',
    'Quais dados monitoram secas no Brasil?',
    'Como a IA detecta queimadas via sensoriamento remoto?',
    'Qual o papel do CEMADEN na prevencao de desastres?',
    'Como dados espaciais apoiam regioes em emergencia?',
]

print('AVALIACAO DO SISTEMA RAG - ORBITAL SENTINEL')
print('=' * 60)
resultados = []
for i, q in enumerate(perguntas_teste, 1):
    print(f'\n[{i}] {q}')
    inicio = time.time()
    resp = query_engine.query(q)
    tempo = time.time() - inicio
    n_f = len(resp.source_nodes) if hasattr(resp, 'source_nodes') else 0
    n_p = len(str(resp).split())
    resultados.append({'tempo': tempo, 'fontes': n_f, 'palavras': n_p})
    print(f'   Tempo: {tempo:.1f}s | Fontes: {n_f} | Palavras: {n_p}')
    print(f'   Previa: {str(resp)[:100]}...')

print('\n' + '=' * 60)
print('SUMARIO')
print(f'  Queries     : {len(resultados)}')
print(f'  Tempo medio : {sum(r["tempo"] for r in resultados)/len(resultados):.1f}s')
print(f'  Fontes/resp : {sum(r["fontes"] for r in resultados)/len(resultados):.1f}')
print(f'  Palavras    : {sum(r["palavras"] for r in resultados)/len(resultados):.0f}')
print('OK RAG: Embedding + FAISS + LLM')
print('OK Interface: Gradio com upload')
print('OK Documentacao: PDF tecnico gerado')
print('=' * 60)


AVALIACAO DO SISTEMA RAG - ORBITAL SENTINEL

[1] Como satelites ajudam a prever enchentes?
   Tempo: 1.6s | Fontes: 4 | Palavras: 247
   Previa: Os satelites ajudam a prever enchentes por meio de tecnologias de sensoriamento remoto, como o senso...

[2] Quais dados monitoram secas no Brasil?
   Tempo: 0.8s | Fontes: 4 | Palavras: 78
   Previa: O produto LST (Land Surface Temperature) do GOES-16 monitora a temperatura da superfície terrestre e...

[3] Como a IA detecta queimadas via sensoriamento remoto?
   Tempo: 0.7s | Fontes: 4 | Palavras: 89
   Previa: A IA detecta queimadas via sensoriamento remoto utilizando algoritmos de deep learning treinados sob...

[4] Qual o papel do CEMADEN na prevencao de desastres?
   Tempo: 1.3s | Fontes: 4 | Palavras: 217
   Previa: O CEMADEN (Centro de Monitoramento e Alertas de Desastres Naturais) desempenha um papel fundamental ...

[5] Como dados espaciais apoiam regioes em emergencia?
   Tempo: 1.2s | Fontes: 4 | Palavras: 195
   Previa: Os dados e